# Lab 1: Unicode Normalization and Tokenization

**Goal:** practice Unicode normalization and build simple tokenizers.

**Dataset used:** `ag_news` (local cache in `datasets/`).

**Time target:** 15–30 minutes.


## Setup

We will locate the shared `datasets/` folder automatically and load a few samples from `ag_news`.


In [61]:
from pathlib import Path
from datasets import load_dataset

def find_datasets_dir(start: Path) -> Path:
    """Find the nearest 'datasets' directory by walking upward."""
    for base in [start] + list(start.parents):
        candidate = base / "datasets"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Could not find datasets directory."
        )

datasets_dir = find_datasets_dir(Path.cwd())
print("Using datasets dir:", datasets_dir)

ag_news = load_dataset(
    "ag_news", cache_dir=str(datasets_dir)
    )

print(
    f"AG news: {ag_news}", 
    "\n", 
    "-"*60, 
    "\n", 
    set(ag_news['train']['label'])
    )

sample_ids = [0, 1, 2]

samples = [ag_news["train"][i] for i in sample_ids]

for s in samples: 
    print(
        f"Lable={s['label']} | text={s['text'][:80]}..."
        )


Using datasets dir: /Users/sorooshshalileh/Teaching /NLP /datasets
AG news: DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
}) 
 ------------------------------------------------------------ 
 {0, 1, 2, 3}
Lable=2 | text=Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall ...
Lable=2 | text=Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment...
Lable=2 | text=Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices p...


## Normalization:

Different inputs can look the same but be encoded differently. Normalization (like NFKC) makes them consistent so tokenizers and models don’t treat them as different words.

In [ ]:
import unicodedata

def normalize_text(text: str) -> str:
    """Normalize a string to Unicode NFKC form.
   
    This applies **Normalization Form KC (NFKC). It reduces superficial variation in text
    by collapsing visually/semantically equivalent forms (e.g., ligatures like "ﬁ" → "fi",
    fullwidth forms → ASCII, and composed vs. decomposed accents).
    Abbrevation: N = Normalizatin, F = Form, K = Compatibility , C = Canonical Composition

    Parameters
    ----------
    text : str
        Input text to normalize. Must be a Python Unicode string.

    Returns
    -------
    str
        The normalized text in NFKC form.

    Notes
    -----
    - NFKC may change string length.
    - Use NFKC when you prefer consistency over preserving typographic
      distinctions; otherwise consider NFC.

    Examples
    --------
    >>> normalize_text("The \ufb01 ligature")
    'The fi ligature'
    >>> normalize_text("nai\u0308ve")
    'naïve'
    """
    return unicodedata.normalize("NFKC", text)

text = "Cafe\u0301 vs Café, the \"\ufb01\" ligature, and fullwidth ＡＢＣ."
normalized_text = normalize_text(text)
print("Raw:", text)
print("Normalized:", normalized_text)
print(len('Cafe\u0301'), len('Café'))

Raw: Café vs Café, the "ﬁ" ligature, and fullwidth ＡＢＣ.
Normalized: Café vs Café, the "fi" ligature, and fullwidth ABC.


In [47]:
unicodedata.normalize("NFC", text)


'Café vs Café, the "ﬁ" ligature, and fullwidth ＡＢＣ.'

**What this example shows:** NFKC makes visually similar text consistent by
- converting decomposed accents (e.g., `Cafe\u0301`) into composed characters (`Café`),
- expanding typographic ligatures (`ﬁ` → `fi`), and
- mapping fullwidth forms (`ＡＢＣ`) to standard ASCII (`ABC`).


In [58]:
import re


def tokenize_whitespace(text: str) -> list[str]:
    """Tokenize text by splitting on whitespace.

    This is the simplest tokenizer: it separates tokens wherever Python's
    whitespace rules apply (spaces, tabs, newlines). It does **not** strip
    punctuation or normalize case.

    Parameters
    ----------
    text : str
        Input string to tokenize.

    Returns
    -------
    list[str]
        List of tokens in the original order.

    Examples
    --------
    >>> tokenize_whitespace("Hello, world!\nNLP")
    ['Hello,', 'world!', 'NLP']
    """
    return text.split()

def tokenize_regex(text: str) -> list[str]:
    """Tokenize text using a simple regex that isolates punctuation.

    This tokenizer returns word-like chunks and keeps punctuation as separate
    tokens. It also keeps simple English contractions together (e.g., "don't",
    "I'm"). It is still a *toy* tokenizer and won’t handle all edge cases.

    Parameters
    ----------
    text : str
        Input string to tokenize.

    Returns
    -------
    list[str]
        List of tokens in the original order.

    Examples
    --------
    >>> tokenize_regex("Hello, world!")
    ['Hello', ',', 'world', '!']
    >>> tokenize_regex("I'm excited :) ")
    ["I'm", 'excited', ':', ')']
    """
    # simpler pattern: r"\w+|[^\w\s]"
    # r"[A-Za-z]+(?:'[A-Za-z]+)?|\d+|[^\w\s]"
    return re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?|\d+|[^\w\s]", text)
example = "Hello, world! NLP is fun :) and I'm exited :)"
print("Whitespace:", tokenize_whitespace(example))
print("Regex:", tokenize_regex(example))


Whitespace: ['Hello,', 'world!', 'NLP', 'is', 'fun', ':)', 'and', "I'm", 'exited', ':)']
Regex: ['Hello', ',', 'world', '!', 'NLP', 'is', 'fun', ':', ')', 'and', "I'm", 'exited', ':', ')']


## Exercise 3 — Compare token counts on AG News

Use your tokenizer(s) on a few AG News samples and compare token counts before and after normalization.


In [64]:
def count_tokens(tokens: list[str]) -> int:
    return len(tokens)


for s in samples:
    raw = s['text']
    norm = normalize_text(raw)
    tokens_raw = tokenize_regex(raw)
    tokens_norm = tokenize_regex(norm)
    print("-"*60)
    print("Raw sample", raw[:80], "...")
    print("Tokens (raw):", count_tokens(raw))
    print("Tokens (norm):", count_tokens(norm))
    if tokens_raw != tokens_norm:
        print("Token lists differ after normalization.")
    else:
        print("Token lists unchanged after normalization.")


------------------------------------------------------------
Raw sample Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall  ...
Tokens (raw): 144
Tokens (norm): 144
Token lists unchanged after normalization.
------------------------------------------------------------
Raw sample Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment ...
Tokens (raw): 266
Tokens (norm): 266
Token lists unchanged after normalization.
------------------------------------------------------------
Raw sample Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices p ...
Tokens (raw): 232
Tokens (norm): 232
Token lists unchanged after normalization.
